# Project Phase 3 Orchestrator Wrapper

Notebook wrapper for the `prototype` agentic application for Peloton Fitness

In [ ]:
from prototype.orchestrator import AgenticOrchestrator

orchestrator = AgenticOrchestrator()
print("Orchestrator initialized.")

## Declare "ask" function to interact with the graph

In [ ]:
def ask(user_query: str, thread_id: str = "default", show_story_output: bool = False, debug: bool = False):
    out = orchestrator.invoke(user_query, thread_id=thread_id)
    print("USER:", user_query)
    print("THREAD:", thread_id)
    print("\nASSISTANT:")
    print(out["response"])
    print(f"\n[routed domain={out['active_domain']} story={out['active_story_id']}]")

    if debug:
        rm = out.get("router_metrics", {})
        print("\nrouter_reason:", out.get("router_reason"))
        print("continuation_score:", rm.get("continuation_score"))
        print("domain_scores:", rm.get("domain_scores"))
        print("margin:", rm.get("margin"))
        print("domain_selected_by:", rm.get("domain_selected_by"))
        print("story_selected_by:", rm.get("story_selected_by"))
        print("domain_guardrail_override:", rm.get("domain_guardrail_override"))

    if show_story_output and out.get("story_output") is not None:
        print("\nstory_output:")
        print(out["story_output"])

    return out

## Orchestrator Graph (LangGraph + Mermaid)
Top-level orchestrator graph visualization

In [ ]:
from IPython.display import Markdown, display, Image

mermaid = orchestrator.get_orchestrator_mermaid()
display(Markdown("```mermaid\n" + mermaid + "\n```"))

try:
    png = orchestrator.get_orchestrator_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


## Membership Fraud Story Graph
Render the LangGraph for `mf_story_1` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.membership_fraud_story1 import get_membership_fraud_story1_mermaid

mf_mermaid = get_membership_fraud_story1_mermaid()
display(Markdown("```mermaid\n" + mf_mermaid + "\n```"))

try:
    # Build an ephemeral graph image from the same definition used by the story module
    from prototype.stories.membership_fraud_story1 import _get_security_graph
    png = _get_security_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "mf_check"

print("1) Missing member_id -> should ask for member_id")
_ = ask("Can you check my most recent suspicious login?", thread_id=thread_id, debug=True)

print("\n2) Provide member_id -> should retrieve alert(s)")
_ = ask("MB001", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Not recognized -> should escalate actions")
_ = ask("I do not recognize that login.", thread_id=thread_id, debug=True)

#print("\n4) Recognized -> should reassure")
#_ = ask("Yes that was me.", thread_id=thread_id, debug=True)


In [ ]:
print("5) Follow-up how-to: password change (should use security help KB)")
_ = ask("I do not recognize it. How do I change my password?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Follow-up how-to: MFA setup (should use security help KB)")
_ = ask("How do I set up multi-factor authentication?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n7) Follow-up how-to: sign out sessions (should use security help KB)")
_ = ask("How do I sign out of all active sessions?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n8) Non-howto check: still recognized flow")
_ = ask("I recognize this device now.", thread_id=thread_id, debug=True)


## Business Marketing Story Graph
Render the LangGraph for `bm_story_1` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.business_marketing_story1 import get_business_marketing_story1_mermaid, _get_business_marketing_graph

bm_mermaid = get_business_marketing_story1_mermaid()
display(Markdown("```mermaid\n" + bm_mermaid + "\n```"))

try:
    png = _get_business_marketing_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "bm_check"

print("1) Campaign + channels + timeframe")
_ = ask("Summarize last month feedback for CAMP105 across app and email and suggest adjustments", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Widened request (no campaign filter)")
_ = ask("Summarize last 8 weeks of marketing feedback and suggest 3 content adjustments", thread_id=thread_id, debug=True)

print("\n3) Sparse/no-match request")
_ = ask("Summarize last week feedback for CAMP999 across social", thread_id=thread_id, debug=True)


In [ ]:
thread_id = "bm_kpi_check"

print("\n4) Definitions intent")
_ = ask("What are CTR, CAC, and ROAS?", thread_id=thread_id, debug=True, show_story_output=True)


In [ ]:
thread_id = "bm_kpi_check"

print("1) Underperformers-only intent (threshold-focused)")
_ = ask("Find underperforming weekly campaign metrics by channel and target segment for last month using CTR, CAC, and ROAS thresholds", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Compare intent + metric scoping (ROAS/CAC only)")
_ = ask("Compare ROAS and CAC by campaign for last quarter and include trend deltas versus prior weeks", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Overview intent (full KPI summary)")
_ = ask("Give me a weekly KPI summary for last month by channel and target segment", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Definitions intent")
_ = ask("What are CTR, CAC, and ROAS?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Concise underperformers request")
_ = ask("Only show underperformers for last month by channel. Keep it brief.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Unsupported-dimension guardrail check")
_ = ask("Show weekly campaign performance by geography for last month", thread_id=thread_id, debug=True, show_story_output=True)


## Data Science Story Graph
Render the LangGraph for `ds_story_2` and run parity-style checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.data_science_story2 import get_data_science_story2_mermaid, _get_data_science_graph

ds_mermaid = get_data_science_story2_mermaid()
display(Markdown("```mermaid\n" + ds_mermaid + "\n```"))

try:
    png = _get_data_science_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "ds_parity"

print("1) Missing member_id -> should ask for member_id")
_ = ask("Am I improving over the past 8 weeks?", thread_id=thread_id, debug=True)

print("\n2) Provide member + trend question")
_ = ask("For MB001, am I improving in my workouts over the last 8 weeks?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Drivers + intensity question")
_ = ask("For MB001 what is driving intensity and do weekdays differ?", thread_id=thread_id, debug=True)

print("\n4) Anomaly question")
_ = ask("For MB001, any unusual drops or spikes in the last 8 weeks?", thread_id=thread_id, debug=True)


## Interactive Multi-Turn Loop
Run this cell for live manual testing. Type `exit` or `quit` to stop.

In [ ]:
active_thread_id = input("Thread ID (default=user_live): " ).strip() or "user_live"
print(f"Interactive multi-turn mode started for thread_id={active_thread_id}. Type 'exit' to stop.")
while True:
    user_text = input("You: " ).strip()
    if user_text.lower() in {"exit", "quit"}:
        print("Stopped interactive mode.")
        break
    if not user_text:
        continue

    out = ask(user_text, thread_id=active_thread_id)
    print(f"(thread={active_thread_id}, domain={out['active_domain']}, story={out['active_story_id']})")
    print('-' * 80)

In [ ]:
active_thread_id = input("Thread ID (default=user_live): " ).strip() or "user_live"
print(f"Interactive multi-turn mode started for thread_id={active_thread_id}. Type 'exit' to stop.")
while True:
    user_text = input("You: " ).strip()
    if user_text.lower() in {"exit", "quit"}:
        print("Stopped interactive mode.")
        break
    if not user_text:
        continue

    out = ask(user_text, thread_id=active_thread_id, debug=True)
    print("pending_slot_type:", out["state"]["pending_slot_type"])
    print("member:", out["state"]["member"])

    print(f"(thread={active_thread_id}, domain={out['active_domain']}, story={out['active_story_id']})")
    print('-' * 80)

## Inspect state

In [ ]:
orchestrator.state